# GCP Pub/Sub — assinatura do BigQuery no seu domínio

Na aula passada vimos tópicos, assinaturas, `pull`, callback e `seek`, no notebook
`gcp-pubsub-v2.ipynb`. Hoje seguimos de onde paramos, a **seção 3**, com uma diferença: em vez da
tabela de anúncios, cada um vai trabalhar com **as tabelas do seu próprio domínio de dados** no
BigQuery.

O caminho de hoje:

1. preparar uma tabela de destino, com o mesmo esquema de uma tabela sua (seção 3);
2. criar o tópico, a DLQ e a assinatura do BigQuery, pela interface (seção 4);
3. provocar um erro e investigar pela DLQ (seção 5);
4. publicar as linhas da sua tabela como mensagens e vê-las chegando no destino (seção 6).

Onde estiver escrito `<assim>`, é para você trocar pelo seu valor. Quase tudo se concentra em uma
célula só, logo no começo.

Referências: [assinaturas do BigQuery](https://cloud.google.com/pubsub/docs/bigquery) e
[tópicos de mensagens inativas](https://cloud.google.com/pubsub/docs/handling-failures).

## Configurações

O ambiente de execução da aula passada já foi encerrado, então é preciso conectar de novo: no canto
superior direito, `Conectar` → `Conectar-se a um ambiente de execução`, modelo `Default`. O passo a
passo com imagens está na primeira seção do `gcp-pubsub-v2.ipynb`.

![conectar-menu](https://raw.githubusercontent.com/robertogyn19/aula-pdm-pubsub/main/imagens-pubsub/img-bq-conectar1-menu.png)

Com a máquina de pé, instale a biblioteca do Pub/Sub. A configuração da assinatura do BigQuery exige a
versão `2.27.0` ou superior. A biblioteca do BigQuery já vem no ambiente — a segunda linha só confere.

In [ ]:
%pip install --upgrade google-cloud-pubsub
%pip install google-cloud-bigquery

In [ ]:
from importlib.metadata import version

# Se esta célula falhar dizendo que o pacote não existe, a instalação acima ainda não foi
# aplicada a esta sessão: reinicie o kernel e rode as duas células de novo.
print("google-cloud-pubsub:", version("google-cloud-pubsub"))
print("google-cloud-bigquery:", version("google-cloud-bigquery"))

In [ ]:
import json
import re
import time
from collections import Counter

import google.auth
from google.api_core.exceptions import AlreadyExists, DeadlineExceeded
from google.cloud import bigquery, pubsub_v1
from google.cloud.pubsub_v1.types import BatchSettings
from google.pubsub_v1 import BigQueryConfig, DeadLetterPolicy, Subscription, Topic

_, project_id = google.auth.default()

publisher = pubsub_v1.PublisherClient()
subscriber = pubsub_v1.SubscriberClient()

print(project_id)

### O seu domínio

**Esta é a célula que você edita.** Troque os três valores entre `<` e `>`:

- `DOMINIO` — um nome curto para o seu domínio, como `vendas` ou `saude`. Ele vira parte do nome de
  todo tópico e assinatura deste notebook, e é o que impede que o seu recurso colida com o de um colega.
  Só letras minúsculas, números e hífen, começando por letra.
- `DATASET` e `TABELA_ORIGEM` — uma tabela **sua** que já existe no BigQuery. Tem que ser tabela, não
  view. Se você tem várias, escolha a mais simples para começar; a outra fica para o exercício do fim.
  Se o seu projeto segue a arquitetura medalhão, escolha uma tabela da camada **silver**: colunas já
  tipadas e limpas, uma linha por registro. A bronze costuma guardar tudo como texto, e aí a seção 5
  não tem coluna para quebrar; a gold é agregada, e uma linha dela não é um evento que alguém
  publicaria. Repare que, na vida real, o caminho é o inverso: é o Pub/Sub que alimenta a bronze. Aqui
  a silver serve só como fonte de mensagens bem formadas.
- `PROJETO_DADOS` — só troque se a sua tabela estiver em outro projeto que não o impresso acima.

Não vamos escrever na sua tabela: ela serve de **modelo** para o esquema e de **fonte** para as
mensagens. Quem recebe os dados é uma cópia vazia, criada na seção 3, com o sufixo `_pubsub`.

In [ ]:
DOMINIO = "<seu-dominio>"  # ex.: "vendas"
DATASET = "<seu_dataset>"  # ex.: "comercial"
TABELA_ORIGEM = "<sua_tabela>"  # ex.: "pedidos"
PROJETO_DADOS = project_id  # troque só se a tabela estiver em outro projeto

# ---------------------------------------------------------------------------------------------
# Daqui para baixo não precisa mexer: os nomes de tudo o que o notebook cria saem dos valores acima.

for nome, valor in {"DOMINIO": DOMINIO, "DATASET": DATASET, "TABELA_ORIGEM": TABELA_ORIGEM}.items():
    if valor.startswith("<"):
        raise ValueError(f"Troque o valor de {nome}: ele ainda está com o texto de exemplo.")

if not re.fullmatch(r"[a-z][a-z0-9-]{1,40}", DOMINIO):
    raise ValueError("DOMINIO: só letras minúsculas, números e hífen, começando por letra.")

tabela_origem = f"{PROJETO_DADOS}.{DATASET}.{TABELA_ORIGEM}"
tabela_destino = f"{PROJETO_DADOS}.{DATASET}.{TABELA_ORIGEM}_pubsub"

# Nomes curtos, que é o que você digita na interface e no gcloud...
nome_topico = f"aula-pdm-{DOMINIO}"
nome_topico_dlq = f"aula-pdm-{DOMINIO}-dlq"
nome_assinatura_dlq = f"aula-pdm-{DOMINIO}-dlq-sub"
nome_assinatura_bq = f"aula-pdm-{DOMINIO}-bq"  # criada pela interface, na 4.3
nome_assinatura_bq_python = f"aula-pdm-{DOMINIO}-bq-python"  # criada por código, na 4.4

# ...e os nomes completos, que é o que a biblioteca do Pub/Sub espera.
topico = publisher.topic_path(project_id, nome_topico)
topico_dlq = publisher.topic_path(project_id, nome_topico_dlq)
assinatura_dlq = subscriber.subscription_path(project_id, nome_assinatura_dlq)
assinatura_bq = subscriber.subscription_path(project_id, nome_assinatura_bq)
assinatura_bq_python = subscriber.subscription_path(project_id, nome_assinatura_bq_python)

print(f"tabela de origem:  {tabela_origem}")
print(f"tabela de destino: {tabela_destino}")
print(f"tópico:            {nome_topico}")
print(f"tópico da DLQ:     {nome_topico_dlq}")
print(f"assinatura do BQ:  {nome_assinatura_bq}")

### Limpando o ambiente (opcional)

Como na aula passada: um tópico ou uma assinatura que já existe mantém a configuração com que foi
criado, e as células de criação só avisam e seguem em frente. Para recomeçar do zero, troque a variável
para `True`. A tabela de destino fica de fora — se quiser apagá-la também, rode no BigQuery Studio
`DROP TABLE` com o nome impresso na célula acima. **A sua tabela de origem nunca é apagada.**

In [ ]:
LIMPAR_AMBIENTE = False

if LIMPAR_AMBIENTE:
    # As assinaturas vêm primeiro: apagar o tópico antes deixaria assinaturas órfãs.
    # Recursos que não existem respondem NOT_FOUND, o que é esperado e pode ser ignorado.
    for nome in [nome_assinatura_bq, nome_assinatura_bq_python, nome_assinatura_dlq]:
        !gcloud pubsub subscriptions delete {nome} --project {project_id} --quiet
    for nome in [nome_topico, nome_topico_dlq]:
        !gcloud pubsub topics delete {nome} --project {project_id} --quiet
else:
    print("Nada foi apagado. Troque LIMPAR_AMBIENTE para True para começar do zero.")

## 3. A tabela de destino no BigQuery

A assinatura do BigQuery escreve em uma tabela que **já existe**, e o esquema dela manda: cada campo do
JSON da mensagem precisa casar, em nome e tipo, com uma coluna da tabela.

Em vez de desenhar esse esquema à mão, como fizemos com os anúncios, vamos copiá-lo da sua tabela. O
`CREATE TABLE ... LIKE` cria uma tabela **vazia** com as mesmas colunas, o mesmo particionamento e o
mesmo clustering da original. Assim a sua tabela fica intocada, e tudo o que aparecer na cópia chegou
pelo Pub/Sub.

Se preferir rodar no BigQuery Studio, o comando é este, com os seus nomes no lugar:

```sql
CREATE TABLE IF NOT EXISTS `<projeto>.<seu_dataset>.<sua_tabela>_pubsub`
LIKE `<projeto>.<seu_dataset>.<sua_tabela>`;
```

A célula abaixo faz o mesmo, já com os nomes da sua configuração.

In [ ]:
bq = bigquery.Client(project=project_id)

bq.query(f"CREATE TABLE IF NOT EXISTS `{tabela_destino}` LIKE `{tabela_origem}`").result()

destino = bq.get_table(tabela_destino)
print(f"{tabela_destino}: {destino.num_rows} linhas\n")
for coluna in destino.schema:
    print(f"  {coluna.name:<30} {coluna.field_type:<10} {coluna.mode}")

Nem todo tipo do BigQuery entra por uma assinatura do Pub/Sub. Três não entram de jeito nenhum:
`GEOGRAPHY`, `INTERVAL` e `RANGE`. E o `GEOGRAPHY` tem um agravante: basta a tabela de destino **ter**
uma coluna desse tipo para **toda** mensagem falhar, inclusive as que nem trazem esse campo.

Por isso a célula abaixo remove essas colunas da cópia. Nas mensagens, o campo continua indo; como a
assinatura vai descartar campos desconhecidos, ele é simplesmente ignorado. Se a sua tabela não tem
nenhum desses tipos, a célula não faz nada.

In [ ]:
TIPOS_SEM_SUPORTE = {"GEOGRAPHY", "INTERVAL", "RANGE"}

sem_suporte = [c.name for c in destino.schema if c.field_type in TIPOS_SEM_SUPORTE]
for nome in sem_suporte:
    # Se a coluna for de particionamento ou clustering, o BigQuery não deixa remover: nesse caso,
    # escolha outra tabela na configuração.
    bq.query(f"ALTER TABLE `{tabela_destino}` DROP COLUMN `{nome}`").result()

destino = bq.get_table(tabela_destino)
print(f"colunas removidas da cópia: {', '.join(sem_suporte) or 'nenhuma'}")

### 3.1. A mensagem é a linha, em JSON

Para a assinatura do BigQuery, uma mensagem é uma linha da tabela escrita em JSON: um objeto cujas
chaves são os nomes das colunas. Para ter mensagens válidas do seu domínio, o jeito mais direto é
pedir ao próprio BigQuery que escreva as suas linhas assim — é o que o `TO_JSON_STRING` faz.

Um aviso de custo: o `LIMIT` reduz o que volta, mas **não** o que o BigQuery lê. Em uma tabela grande,
a consulta é cobrada pela tabela inteira (só das colunas lidas). Para as tabelas da disciplina, isso não
chega a ser um problema.

In [ ]:
LIMITE = 10  # quantas linhas da sua tabela viram mensagens na seção 6

consulta = f"SELECT TO_JSON_STRING(t) AS linha FROM `{tabela_origem}` AS t LIMIT {LIMITE}"
linhas_json = [registro.linha for registro in bq.query(consulta).result()]

if not linhas_json:
    raise ValueError(f"A tabela {tabela_origem} está vazia: escolha outra na configuração.")

print(f"{len(linhas_json)} linhas lidas de {tabela_origem}\n")
print(json.dumps(json.loads(linhas_json[0]), indent=2, ensure_ascii=False)[:1500])

Olhe a saída acima com atenção, porque é **exatamente** o que vai trafegar pelo tópico. Repare em
como cada tipo foi escrito: datas e horários viram texto, `ARRAY` vira lista, `STRUCT` vira um objeto
aninhado, `NULL` vira `null`. `NUMERIC`, `BIGNUMERIC` e inteiros muito grandes também viram texto — é
o jeito de não perder precisão num número JSON — e `BYTES` vira base64. Tudo isso a assinatura
entende na volta.

A exceção é a coluna do tipo `JSON`. O `TO_JSON_STRING` a escreve como um objeto aninhado, mas a
assinatura exige que o valor chegue como **texto** contendo o JSON. A célula abaixo faz essa conversão.
É o primeiro exemplo de algo que vai se repetir: quem publica precisa adaptar o dado ao formato que o
destino aceita. Se a sua tabela não tem coluna `JSON`, a célula não muda nada.

In [ ]:
colunas_json = [c.name for c in destino.schema if c.field_type == "JSON"]


def ajustar_para_assinatura(linha):
    registro = json.loads(linha)
    for coluna in colunas_json:
        if registro.get(coluna) is not None:
            registro[coluna] = json.dumps(registro[coluna], ensure_ascii=False)
    return json.dumps(registro, ensure_ascii=False)


if colunas_json:
    linhas_json = [ajustar_para_assinatura(linha) for linha in linhas_json]
print(f"colunas JSON convertidas para texto: {', '.join(colunas_json) or 'nenhuma'}")

## 4. Integração entre BigQuery e Pub/Sub

Primeiro o tópico que vai receber as mensagens do seu domínio. Um dia de retenção é mais que suficiente
para a aula, e permite usar o `seek` da seção 2.3 se precisar reprocessar.

In [ ]:
topico_obj = Topic({
    "name": topico,
    "message_retention_duration": "86400s",  # 1 dia
})
try:
    publisher.create_topic(request=topico_obj)
    print(f"Tópico '{topico}' criado com sucesso")
except AlreadyExists:
    print(f"O tópico '{topico}' já existe e mantém a configuração anterior")

### 4.1. Permissões do Pub/Sub para o BigQuery

Quem escreve na tabela de destino não é você: é o **agente de serviço do Pub/Sub**, uma conta que a
Google cria no projeto, no formato `service-<numero-do-projeto>@gcp-sa-pubsub.iam.gserviceaccount.com`.
Ela precisa do papel `BigQuery Data Editor` no projeto onde a tabela está.

Lembrete da aula passada: o **número** do projeto é diferente do **ID**. Trocar os dois é o erro mais
comum aqui, e só aparece bem depois, com a assinatura em estado de erro.

Se o comando de concessão responder que você não tem permissão, não é problema seu: em projeto
compartilhado, quem administra o projeto já concedeu antes da aula. Siga em frente.

In [ ]:
numero_projeto = !gcloud projects describe {project_id} --format="value(projectNumber)"
conta_pubsub = f"service-{numero_projeto[0]}@gcp-sa-pubsub.iam.gserviceaccount.com"
conta_pubsub

In [ ]:
!gcloud projects add-iam-policy-binding {PROJETO_DADOS} --member="serviceAccount:{conta_pubsub}" --role="roles/bigquery.dataEditor" --condition=None --format="value(etag)" 

### 4.2. Tópico e assinatura da DLQ

A DLQ, a fila de mensagens mortas, é um tópico comum. O que o torna especial é a assinatura do BigQuery
apontar para ele: a mensagem que falhar na inserção depois de algumas tentativas é desviada para cá, em
vez de ficar sendo reentregue para sempre.

E um tópico sem assinatura não guarda nada — mensagem publicada em tópico sem assinatura é descartada
na hora. Por isso criamos as duas coisas juntas.

Repare no `ack_deadline_seconds`. Na seção 2.1 o prazo de confirmação era de 10 segundos, o padrão. Na
DLQ quem lê é uma pessoa investigando um erro, não um programa, e 10 segundos não dão nem para ler a
mensagem. Aqui usamos o máximo, 10 minutos.

In [ ]:
try:
    publisher.create_topic(name=topico_dlq)
    print(f"Tópico '{topico_dlq}' criado com sucesso")
except AlreadyExists:
    print(f"O tópico '{topico_dlq}' já existe")

try:
    subscriber.create_subscription(
        name=assinatura_dlq,
        topic=topico_dlq,
        ack_deadline_seconds=600,  # 10 minutos: quem lê a DLQ é gente, não programa
    )
    print(f"Assinatura '{assinatura_dlq}' criada com sucesso")
except AlreadyExists:
    print(f"A assinatura '{assinatura_dlq}' já existe")

### 4.3. Criação da assinatura pela interface gráfica

Agora a assinatura que leva as mensagens para a tabela. A célula abaixo imprime os valores que você vai
digitar no formulário — deixe a saída dela à vista enquanto preenche. As imagens são da aula passada,
com os nomes dos anúncios; use os **seus**.

In [ ]:
print("Valores para o formulário de 'Criar assinatura'\n")
print(f"  Código da assinatura ........ {nome_assinatura_bq}")
print(f"  Tópico ...................... projects/{project_id}/topics/{nome_topico}")
print(f"  Tipo de envio ............... Gravar no BigQuery")
print(f"  Projeto ..................... {PROJETO_DADOS}")
print(f"  Conjunto de dados ........... {DATASET}")
print(f"  Tabela ...................... {TABELA_ORIGEM}_pubsub")
print(f"  Conta de serviço ............ padrão para agente de serviço do Pub/Sub")
print(f"  Esquema ..................... Usar o esquema da tabela")
print(f"  Descartar campos desconhecidos  marcado")
print(f"  Mensagens inativas .......... ativado")
print(f"  Tópico de mensagens inativas  projects/{project_id}/topics/{nome_topico_dlq}")
print(f"  Máximo de tentativas ........ 5")

Na página de [assinaturas do Pub/Sub](https://console.cloud.google.com/cloudpubsub/subscription),
clique em `Criar assinatura`.

1. **Código da assinatura**: o `aula-pdm-<seu-dominio>-bq` impresso acima.
2. **Tópico**: o `aula-pdm-<seu-dominio>`, criado no começo da seção 4.
3. **Tipo de envio**: `Gravar no BigQuery`. É esta opção que faz o Pub/Sub inserir na tabela sozinho,
   sem nenhum código de consumo do nosso lado — nada de assinante, nada de `ack`.
4. a 6. **Projeto, conjunto de dados e tabela**: a tabela **`_pubsub`**, a cópia vazia da seção 3. Não
   a sua tabela de origem.
   **Conta de serviço**, logo abaixo da tabela: tem que estar em `Conta de serviço (padrão para agente
   de serviço do Pub/Sub)`. Se estiver com a conta do Compute Engine, a que termina em
   `-compute@developer.gserviceaccount.com`, o formulário reclama que faltam `bigquery.tables.get` e
   `bigquery.tables.updateData`. Não conceda nada a ela: troque a opção. A permissão da 4.1 foi dada ao
   agente do Pub/Sub, não a essa conta.
7. A confirmação em verde é a prova de que a permissão da 4.1 foi concedida. Se aparecer um aviso no
   lugar dela, a assinatura até é criada, mas fica em estado de erro e nada chega na tabela.

Nas opções de esquema, marque **Usar o esquema da tabela** e **Descartar campos desconhecidos**. A
primeira faz o Pub/Sub casar os campos do JSON com as colunas; a segunda faz um campo a mais na
mensagem ser ignorado, em vez de derrubar a mensagem inteira.

![criar-assinatura-bq](https://raw.githubusercontent.com/robertogyn19/aula-pdm-pubsub/main/imagens-pubsub/img-sub-bq-form.png)

Mais abaixo, no tratamento de falhas:

1. **Mensagens inativas** liga o encaminhamento para a DLQ.
2. O tópico é o `aula-pdm-<seu-dominio>-dlq`, da seção 4.2.

O `Máximo de tentativas de entrega` é quantas vezes o Pub/Sub insiste antes de desistir e mandar a
mensagem para a DLQ. Cinco é o mínimo permitido, e é o que queremos: quanto menor, mais rápido a seção 5
mostra o erro.

![criar-assinatura-falhas](https://raw.githubusercontent.com/robertogyn19/aula-pdm-pubsub/main/imagens-pubsub/img-sub-bq-falhas.png)

Criar a assinatura não basta. Abra a aba `Mensagens inativas` dela: o tópico aparece resolvido, mas
as duas verificações seguintes vêm com alerta.

1. `Conceder o papel de Editor` — permite ao Pub/Sub **publicar** no tópico da DLQ as mensagens que
   falharam.
2. `Conceder o papel de Assinante` — permite a ele **confirmar** as mensagens desta assinatura ao
   encaminhá-las.

Os dois botões só aparecem depois que a assinatura existe, e é por isso que não cabiam junto com a
permissão do BigQuery na 4.1.

**Enquanto os dois alertas estiverem de pé, a mensagem com erro simplesmente não chega na DLQ**, sem
nenhum aviso — e a seção 5 não vai ter o que mostrar.

![permissoes-dlq](https://raw.githubusercontent.com/robertogyn19/aula-pdm-pubsub/main/imagens-pubsub/img-sub-bq-dlq-permissoes.png)

### 4.4. A mesma assinatura, por código

Tudo o que o formulário pediu aparece aqui com outro nome: a tabela e as duas opções de esquema viram o
`BigQueryConfig`; a DLQ e o número de tentativas viram o `DeadLetterPolicy`.

A criação está desligada de propósito. Duas assinaturas do BigQuery no mesmo tópico, apontando para a
mesma tabela, escrevem **cada mensagem duas vezes** — um tópico entrega uma cópia para cada assinatura
sua. Se quiser rodar, troque a flag para `True`, sabendo que as contagens da seção 6 vão dobrar.

In [ ]:
assinatura_bq_obj = Subscription({
    "name": assinatura_bq_python,
    "topic": topico,
    "bigquery_config": BigQueryConfig({
        "table": tabela_destino,  # projeto.dataset.tabela
        "use_table_schema": True,  # casa os campos do JSON com as colunas da tabela
        "drop_unknown_fields": True,  # campo que não existe na tabela é ignorado, não é erro
    }),
    "dead_letter_policy": DeadLetterPolicy({
        "dead_letter_topic": topico_dlq,
        "max_delivery_attempts": 5,
    }),
})
assinatura_bq_obj

In [ ]:
CRIAR_ASSINATURA_POR_CODIGO = False

if CRIAR_ASSINATURA_POR_CODIGO:
    try:
        subscriber.create_subscription(request=assinatura_bq_obj)
        print(f"Assinatura '{assinatura_bq_python}' criada com sucesso")
    except AlreadyExists:
        print(f"A assinatura '{assinatura_bq_python}' já existe")

    # As mesmas duas permissões dos botões da aba 'Mensagens inativas', agora por comando.
    # Só podem ser concedidas depois que o tópico e a assinatura existem.
    !gcloud pubsub topics add-iam-policy-binding {topico_dlq} --member="serviceAccount:{conta_pubsub}" --role="roles/pubsub.publisher"
    !gcloud pubsub subscriptions add-iam-policy-binding {assinatura_bq_python} --member="serviceAccount:{conta_pubsub}" --role="roles/pubsub.subscriber"
else:
    print("Nada foi criado. Troque CRIAR_ASSINATURA_POR_CODIGO para True para criar por código.")

## 5. Troubleshooting pela DLQ

A assinatura do BigQuery não tem um assinante escrevendo `print` quando algo dá errado. Quando uma
mensagem não entra na tabela, **nenhuma célula falha e nenhum erro aparece na tela**: a mensagem é
reentregue, falha de novo, e depois da quinta tentativa é desviada para a DLQ, com o motivo anotado nos
atributos. Investigar uma assinatura do BigQuery é, quase sempre, ler a DLQ.

Nesta seção vamos percorrer o ciclo inteiro:

1. conferir que a assinatura está saudável;
2. provocar um erro com uma mensagem do seu domínio;
3. ler a DLQ e entender o motivo;
4. corrigir, reenviar e ver a linha chegar na tabela.

### 5.1. A assinatura está saudável?

Antes de procurar erro em mensagem, confira a assinatura. O campo `state` responde a primeira pergunta:

- `ACTIVE` — a assinatura consegue escrever na tabela. Erro daqui em diante é de mensagem.
- `RESOURCE_ERROR` — o problema é da assinatura, não das mensagens: a tabela não existe, ou o agente de
  serviço não tem a permissão da 4.1. Nesse estado **nada** vai para a DLQ; as mensagens ficam retidas
  na assinatura até o problema ser resolvido.

A saída também mostra a tabela e a DLQ configuradas — vale conferir que é a `_pubsub`, e não a origem.

In [ ]:
!gcloud pubsub subscriptions describe {nome_assinatura_bq} --project {project_id} --format="yaml(state,topic,bigqueryConfig,deadLetterPolicy)" 

A segunda conferência é a que falha em silêncio: as duas permissões da aba `Mensagens inativas`. Se a
política abaixo não listar o agente de serviço com o papel `roles/pubsub.publisher`, as mensagens com
erro não chegam na DLQ — e a DLQ vazia vai parecer notícia boa, quando não é.

In [ ]:
print(f"agente de serviço: {conta_pubsub}\n")
!gcloud pubsub topics get-iam-policy {nome_topico_dlq} --project {project_id} --format="yaml(bindings)" 

### 5.2. Provocando um erro

Vamos montar uma mensagem inválida a partir de uma linha **verdadeira** da sua tabela: pegamos a
primeira linha da seção 3.1 e trocamos o valor de uma coluna não textual — um número, uma data, um
booleano — por um texto qualquer. É o erro mais comum na vida real: o produtor manda um campo com o
tipo errado.

A célula escolhe a coluna sozinha. Se quiser quebrar outra, escreva o nome dela em
`COLUNA_PARA_QUEBRAR`.

In [ ]:
COLUNA_PARA_QUEBRAR = None  # ou o nome de uma coluna sua, ex.: "valor_total"

TIPOS_NAO_TEXTUAIS = {
    "INTEGER", "INT64", "FLOAT", "FLOAT64", "NUMERIC", "BIGNUMERIC",
    "BOOLEAN", "BOOL", "DATE", "DATETIME", "TIMESTAMP", "TIME",
}

linha_valida = json.loads(linhas_json[0])

if COLUNA_PARA_QUEBRAR is None:
    candidatas = [
        c.name for c in destino.schema
        if c.field_type in TIPOS_NAO_TEXTUAIS and c.mode != "REPEATED"
    ]
    if not candidatas:
        raise ValueError("A tabela só tem colunas de texto: escolha outra em COLUNA_PARA_QUEBRAR.")
    COLUNA_PARA_QUEBRAR = candidatas[0]

valor_original = linha_valida.get(COLUNA_PARA_QUEBRAR)
mensagem_invalida = {**linha_valida, COLUNA_PARA_QUEBRAR: "valor-inválido"}

print(f"coluna quebrada: {COLUNA_PARA_QUEBRAR}  (valor original: {valor_original!r})\n")
print("Copie o JSON abaixo:\n")
print(json.dumps(mensagem_invalida, indent=2, ensure_ascii=False))

Agora publique essa mensagem **pelo console**: abra o seu tópico `aula-pdm-<seu-dominio>` na
[página de tópicos](https://console.cloud.google.com/cloudpubsub/topic/list), vá até a aba `Mensagens`
e clique em `Publicar mensagem`.

![publicar-mensagem](https://raw.githubusercontent.com/robertogyn19/aula-pdm-pubsub/main/imagens-pubsub/img-v2-pubsub-topic-publish-msg.png)

Cole o JSON no campo `Corpo da mensagem` e clique em `Publicar`.

![publicar-mensagem](https://raw.githubusercontent.com/robertogyn19/aula-pdm-pubsub/main/imagens-pubsub/img-v2-pubsub-topic-publish-msg2.png)

Se o console der problema, a célula abaixo faz a mesma publicação por código. Use uma ou outra, não as
duas.

In [ ]:
PUBLICAR_POR_CODIGO = False

if PUBLICAR_POR_CODIGO:
    corpo = json.dumps(mensagem_invalida, ensure_ascii=False).encode("utf-8")
    print("publicada:", publisher.publish(topico, corpo, origem="secao-5").result())

### 5.3. Lendo a DLQ

As cinco tentativas levam alguns segundos. A célula abaixo usa o `pull` da seção 2.1, e não o
`subscribe` da 2.2, por dois motivos: ela não prende o kernel, e ela **não confirma** nada — lemos,
analisamos, e só confirmamos quando tivermos decidido o que fazer com cada mensagem.

Se ela voltar vazia depois de um minuto, volte à 5.1: é quase sempre a permissão da DLQ.

In [ ]:
ATRIBUTO_ERRO = "CloudPubSubDeadLetterSourceDeliveryErrorMessage"

recebidas_dlq = []
for tentativa in range(1, 7):
    try:
        resposta = subscriber.pull(subscription=assinatura_dlq, max_messages=50, timeout=10)
        recebidas_dlq = list(resposta.received_messages)
    except DeadlineExceeded:
        recebidas_dlq = []

    print(f"tentativa {tentativa}: {len(recebidas_dlq)} mensagem(ns)")
    if recebidas_dlq:
        break
    time.sleep(10)

for recebida in recebidas_dlq:
    atributos = dict(recebida.message.attributes)
    print("\n" + "-" * 100)
    print(f"ERRO: {atributos.pop(ATRIBUTO_ERRO, '(sem mensagem de erro)')}\n")
    for chave, valor in sorted(atributos.items()):
        print(f"  {chave}: {valor}")
    print(f"\n  corpo: {recebida.message.data.decode('utf-8')[:300]}")

Cada mensagem que cai na DLQ chega com o corpo **original**, intacto, e com atributos que o Pub/Sub
acrescenta. Vale ler um por um:

| Atributo | O que diz |
|---|---|
| `CloudPubSubDeadLetterSourceDeliveryErrorMessage` | **o motivo** — a resposta do BigQuery ao tentar inserir. É o que você procura. |
| `CloudPubSubDeadLetterSourceDeliveryCount` | quantas vezes o Pub/Sub tentou: as 5 da configuração |
| `CloudPubSubDeadLetterSourceSubscription` | de qual assinatura a mensagem veio — uma DLQ pode servir a várias |
| `CloudPubSubDeadLetterSourceTopicPublishTime` | quando a mensagem foi publicada no tópico de origem |

Os atributos que o produtor colocou na mensagem também continuam lá. É por isso que vale publicar com
atributos como `origem`: na hora do erro, eles dizem de onde a mensagem veio.

Algumas mensagens de erro que vocês podem encontrar, e o que fazer com cada uma:

| O erro diz... | A causa | O que fazer |
|---|---|---|
| `is not compatible with the passed in JSON` | um campo com o tipo errado — o da 5.2 | corrigir o produtor |
| `Tried to parse invalid JSON` | o corpo nem é um JSON | corrigir o produtor; não há o que reaproveitar |
| `To write data to a JSON field it must be a valid JSON string` | coluna `JSON` recebeu um objeto em vez de texto | a conversão da seção 3.1 |
| `BigQuery field type GEOGRAPHY` | a tabela de destino tem coluna `GEOGRAPHY` — e aí **toda** mensagem falha | remover a coluna, como na seção 3 |
| fala de um campo obrigatório (`required`) | o JSON não trouxe uma coluna `REQUIRED` | mandar o campo, ou deixar a coluna `NULLABLE` |
| nada — a DLQ está vazia e a tabela também | a permissão da DLQ ou o `state` da assinatura | voltar à 5.1 |

E um que **não** aparece aqui: um campo a mais, que não existe na tabela. Com `Descartar campos
desconhecidos` marcado, ele é ignorado e a linha entra normalmente.

Guarde o formato do erro: é o mesmo raciocínio para qualquer esquema, de qualquer domínio.

### 5.4. Corrigindo e reenviando

Investigar a DLQ só termina quando a mensagem chega onde deveria. Como sabemos o que quebramos,
conseguimos consertar: devolvemos o valor original à coluna e publicamos a mensagem de novo **no tópico
de origem**. A assinatura do BigQuery recebe como se fosse nova.

Em produção, o conserto é o código que você escreveria depois de ler o erro — às vezes corrigir o
produtor e reprocessar, às vezes descartar. O que não muda é a ordem: **reenviar primeiro, confirmar na
DLQ depois**. Confirmar antes e falhar no reenvio perde a mensagem de vez.

In [ ]:
reenviadas, descartadas = 0, 0

for recebida in recebidas_dlq:
    try:
        mensagem = json.loads(recebida.message.data.decode("utf-8"))
    except json.JSONDecodeError:
        descartadas += 1  # nem é JSON: não há o que corrigir
        continue

    if mensagem.get(COLUNA_PARA_QUEBRAR) == "valor-inválido":
        mensagem[COLUNA_PARA_QUEBRAR] = valor_original

    corpo = json.dumps(mensagem, ensure_ascii=False).encode("utf-8")
    publisher.publish(topico, corpo, origem="reprocessamento-dlq").result()
    reenviadas += 1

# Só depois de reenviar é que confirmamos na DLQ.
if recebidas_dlq:
    subscriber.acknowledge(
        subscription=assinatura_dlq,
        ack_ids=[recebida.ack_id for recebida in recebidas_dlq],
    )
recebidas_dlq = []

print(f"{reenviadas} reenviadas ao tópico, {descartadas} descartadas, DLQ confirmada")

A escrita da assinatura leva alguns instantes. Rode a consulta: se vier zero, espere uns segundos e
rode de novo. A mesma consulta funciona no BigQuery Studio, com o nome da sua tabela `_pubsub`.

In [ ]:
consulta = f"SELECT COUNT(*) AS total FROM `{tabela_destino}`"
print(f"{tabela_destino}: {list(bq.query(consulta).result())[0].total} linha(s)")

## 6. Publicando os seus dados no tópico

Com a assinatura testada nos dois caminhos — o do erro e o do acerto —, é hora de mandar volume: as
linhas da seção 3.1, cada uma como uma mensagem.

Publicar uma mensagem por vez custaria uma chamada de rede por linha. O cliente do Pub/Sub sabe
agrupar mensagens em lotes, e o `BatchSettings` define quando um lote é fechado: o que acontecer
primeiro entre os três limites abaixo. Usamos um cliente separado, para não mexer no `publisher` das
seções anteriores.

In [ ]:
batch_settings = BatchSettings(
    max_bytes=5 * 1024 * 1024,  # no máximo 5MB por lote
    max_messages=1000,  # até 1000 mensagens por lote
    max_latency=0.25,  # envia a cada 250ms
)
publisher_lote = pubsub_v1.PublisherClient(batch_settings=batch_settings)

O `publish` é assíncrono e devolve um `Future` — ele não espera a confirmação do servidor. Por isso a
célula abaixo termina quase instantaneamente: o envio de verdade acontece em segundo plano.

Os argumentos nomeados depois do corpo viram **atributos** da mensagem: metadados que viajam ao lado do
corpo, não entram na tabela e voltam intactos se a mensagem cair na DLQ. O `linha` diz qual linha da
amostra a mensagem é — na DLQ, é o que permite achar a linha problemática na origem.

In [ ]:
futures = [
    publisher_lote.publish(topico, linha.encode("utf-8"), origem="secao-6", linha=str(idx))
    for idx, linha in enumerate(linhas_json)
]
print(f"{len(futures)} publicações enfileiradas")

É o `result()` que espera a confirmação de cada publicação. Se alguma falhar, é aqui que a exceção
aparece. Repare que "publicada" quer dizer que o **tópico** aceitou a mensagem — não que ela entrou na
tabela. Essa segunda parte é com a assinatura.

In [ ]:
for fut in futures:
    fut.result(timeout=60)

print(f"{len(futures)} mensagens publicadas em {topico}")

### 6.1. Conferindo no BigQuery e na DLQ

Compare: a tabela `_pubsub` deve ter as linhas publicadas aqui, mais a da seção 5.4. A assinatura
escreve em segundo plano, então logo depois da publicação é normal a contagem vir menor — espere uns
segundos e rode de novo antes de concluir que falta alguma coisa.

In [ ]:
consulta = f"SELECT COUNT(*) AS total FROM `{tabela_destino}`"
total = list(bq.query(consulta).result())[0].total
print(f"{tabela_destino}: {total} linha(s) — publicadas agora: {len(futures)}")

Se faltar linha, a resposta está na DLQ. Com volume, olhar mensagem por mensagem não funciona: a
célula abaixo lê tudo o que estiver lá e **agrupa pelo motivo do erro**. Dez mensagens com o mesmo
erro — ou dez mil — são um problema só, e é assim que se investiga uma falha em massa.

Ela confirma o que leu, para esvaziar a DLQ: as linhas continuam na sua tabela de origem, e o atributo
`linha` diz quais eram.

In [ ]:
time.sleep(15)  # as cinco tentativas de cada mensagem levam alguns segundos

erros, linhas_com_erro, ack_ids = Counter(), [], []
while True:
    try:
        resposta = subscriber.pull(subscription=assinatura_dlq, max_messages=500, timeout=10)
    except DeadlineExceeded:
        break
    if not resposta.received_messages:
        break
    for recebida in resposta.received_messages:
        atributos = recebida.message.attributes
        erros[atributos.get(ATRIBUTO_ERRO, "(sem mensagem de erro)")] += 1
        linhas_com_erro.append(atributos.get("linha", "?"))
        ack_ids.append(recebida.ack_id)

if ack_ids:
    for inicio in range(0, len(ack_ids), 1000):
        subscriber.acknowledge(subscription=assinatura_dlq, ack_ids=ack_ids[inicio:inicio + 1000])

print(f"{sum(erros.values())} mensagem(ns) na DLQ\n")
for erro, quantidade in erros.most_common():
    print(f"{quantidade:>5} × {erro}")
if linhas_com_erro:
    print(f"\nlinhas da amostra que falharam: {', '.join(linhas_com_erro[:20])}")

Se a DLQ veio vazia e a contagem bateu, o pipeline está completo: **nenhuma linha de código de
inserção foi escrita**. Quem escreveu na tabela foi a assinatura que você criou pelo formulário.

Se veio com erros, procure o motivo na tabela da seção 5.3 e pense em onde ele nasce. A correção quase
sempre é no produtor: converter aquela coluna antes de publicar, como a seção 3.1 fez com as colunas
`JSON` e a 5.4 fez com uma mensagem só.

## 7. Para continuar

Se quiser ir além, o próximo passo é o notebook
[`gcp-pipeline-anuncios.ipynb`](https://github.com/robertogyn19/aula-pdm-pubsub/blob/main/gcp-pipeline-anuncios.ipynb).
Ele segue um anúncio de imóvel pelo pipeline inteiro: a coleta na API do Chaves na Mão, o texto indo
pelo Pub/Sub até o BigQuery, como vocês fizeram hoje, e as fotos indo para o Cloud Storage, de onde o
Gemini extrai o que o texto não diz — o cômodo e o tipo de piso.

Para carregá-lo no BigQuery Studio, use a opção `URL` do upload, como na aula passada:

```
https://raw.githubusercontent.com/robertogyn19/aula-pdm-pubsub/main/gcp-pipeline-anuncios.ipynb
```

Ele usa o tópico `aula-pdm-anuncios`, a tabela `aula_pdm.anuncios` e o bucket criados pelo
`gcp-pubsub-v2.ipynb`, então confira antes que eles ainda existem no seu projeto — e que o tópico ainda
tem a assinatura do BigQuery. Tópico sem assinatura descarta o que recebe.

## Encerrando

Antes de sair, **exclua o ambiente de execução**: `Conectar` → `Gerenciar sessões`, e encerre a sua
sessão. Ele fica faturando enquanto estiver de pé, mesmo sem ninguém executando nada.

Os tópicos, as assinaturas e a tabela `_pubsub` podem ficar — não custam nada parados. Para apagar os
recursos do Pub/Sub, use a célula *Limpando o ambiente*, lá no começo.